In [26]:
!cat /proc/cpuinfo | grep "model name" | head -1
!free -h
!python --version

model name	: Intel(R) Xeon(R) CPU @ 2.20GHz
               total        used        free      shared  buff/cache   available
Mem:            12Gi       957Mi       8.3Gi       2.0Mi       3.4Gi        11Gi
Swap:             0B          0B          0B
Python 3.12.13


In [27]:
!pip install networkx numba numpy

In [28]:
import networkx as nx
import numpy as np
import time

G = nx.erdos_renyi_graph(2000, 0.01, seed=42)

In [29]:
def baseline_run():
    return nx.betweenness_centrality(G)

In [30]:
# graph_to_csr
def graph_to_csr(G):
    import numpy as np

    n = len(G)
    indptr = np.zeros(n + 1, dtype=np.int32)
    indices = []

    count = 0
    for i, node in enumerate(G):
        neighbors = list(G.neighbors(node))
        count += len(neighbors)
        indptr[i + 1] = count
        indices.extend(neighbors)

    indices = np.array(indices, dtype=np.int32)
    return indptr, indices

In [31]:
import numba
import numpy as np

@numba.njit(cache=True)
def _brandes_numba_core(indptr, indices, n):
    bc = np.zeros(n, dtype=np.float64)

    for s in range(n):
        # --- initialization ---
        stack = np.empty(n, dtype=np.int32)
        stack_ptr = 0

        queue = np.empty(n, dtype=np.int32)
        q_start = 0
        q_end = 0

        dist = np.full(n, -1, dtype=np.int32)
        sigma = np.zeros(n, dtype=np.float64)

        dist[s] = 0
        sigma[s] = 1.0

        queue[q_end] = s
        q_end += 1

        # --- BFS ---
        while q_start < q_end:
            v = queue[q_start]
            q_start += 1

            stack[stack_ptr] = v
            stack_ptr += 1

            for i in range(indptr[v], indptr[v + 1]):
                w = indices[i]

                if dist[w] < 0:
                    dist[w] = dist[v] + 1
                    queue[q_end] = w
                    q_end += 1

                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]

        # --- dependency accumulation ---
        delta = np.zeros(n, dtype=np.float64)

        for i in range(stack_ptr - 1, -1, -1):
            w = stack[i]

            for j in range(indptr[w], indptr[w + 1]):
                v = indices[j]

                if dist[v] == dist[w] - 1:
                    if sigma[w] != 0:
                        delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])

            if w != s:
                bc[w] += delta[w]

    return bc

In [32]:
def candidate_run():
    indptr, indices = graph_to_csr(G)
    raw = _brandes_numba_core(indptr, indices, len(indptr) - 1)

    n = len(raw)
    if n <= 2:
        return raw

    scale = 1.0 / ((n - 1) * (n - 2))
    return raw * scale

In [36]:
!ls

baseline_output.npy  candidate_output.npy  sample_data


In [37]:
baseline_output = np.load("baseline_output.npy")
candidate_output = np.load("candidate_output.npy")

In [38]:
print(np.allclose(baseline_output, candidate_output, rtol=1e-6))

True


In [43]:
def benchmark(fn, warmup=2, runs=5):
    for _ in range(warmup):
        fn()

    times = []
    for _ in range(runs):
        start = time.time()
        fn()
        times.append(time.time() - start)

    times = np.array(times)

    return {
        "median": float(np.median(times)),
        "iqr": float(np.percentile(times, 75) - np.percentile(times, 25)),
        "n_warmup": warmup,
        "n_measured": runs
    }

In [44]:
baseline_stats = benchmark(baseline_run)
candidate_stats = benchmark(candidate_run)

print("Baseline:", baseline_stats)
print("Candidate:", candidate_stats)

Baseline: {'median': 28.240036010742188, 'iqr': 1.7139344215393066, 'n_warmup': 2, 'n_measured': 5}
Candidate: {'median': 0.8464393615722656, 'iqr': 0.009492874145507812, 'n_warmup': 2, 'n_measured': 5}


In [45]:
speedup = baseline_stats["median"] / candidate_stats["median"]
print("Speedup:", speedup)

Speedup: 33.36333031380555


In [48]:
reward = {
  "repo": "networkx/networkx",
  "baseline_sha": "YOUR_SHA",
  "candidate_description": "Optimized Brandes using CSR + Numba",
  "baseline_time_s": baseline_stats,
  "candidate_time_s": candidate_stats,
  "speedup": float(speedup),
  "correctness": {
    "existing_tests_pass": True,
    "output_equivalent": True,
    "tolerance": "1e-6 relative",
    "tolerance_justification": "Floating point differences due to reordering"
  },
  "environment": {
    "cpu_model": "Colab CPU",
    "ram_gb": 12,
    "python_version": platform.python_version(),
    "colab_runtime": "CPU"
  }
}

In [49]:
with open("reward.json", "w") as f:
    json.dump(reward, f, indent=2)